# RGBD perception

This notebook visualizes the RGBD feed and runs a simple object-visible check. It uses `sdk_client.Robot.get_rgbd()` for frames, then combines color thresholding and depth coverage inside a region of interest.


In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Import display, image, and robot helpers.


In [ ]:
import base64
import json
import time

import cv2
import ipywidgets as widgets
import numpy as np
from IPython.display import display

from sdk_client import Robot


Create the robot RGBD client. Override `G1_RGBD_HOST` and `G1_RGBD_PORT` if the image server runs elsewhere.


In [ ]:
RGBD_HOST = os.environ.get("G1_RGBD_HOST", "192.168.2.41")
RGBD_PORT = int(os.environ.get("G1_RGBD_PORT", "5555"))
RGBD_TOPIC = os.environ.get("G1_RGBD_TOPIC", "")
robot = Robot(
    iface=IFACE,
    domain_id=DOMAIN_ID,
    safety_boot=False,
    recover_dev_mode_on_init=False,
    auto_start_sensors=False,
    rgbd_host=RGBD_HOST,
    rgbd_port=RGBD_PORT,
    rgbd_topic=RGBD_TOPIC,
)
print(f"RGBD client ready for tcp://{RGBD_HOST}:{RGBD_PORT} topic={RGBD_TOPIC!r}")


Helper functions for HTML image display and a simple visibility detector.


In [ ]:
def jpeg_data_url(bgr):
    ok, buf = cv2.imencode(".jpg", bgr)
    if not ok:
        return ""
    payload = base64.b64encode(buf.tobytes()).decode("ascii")
    return f"data:image/jpeg;base64,{payload}"


def colorize_depth(depth_m, max_depth_m=4.0):
    valid = np.isfinite(depth_m) & (depth_m > 0)
    disp = np.zeros(depth_m.shape[:2], dtype=np.uint8)
    disp[valid] = np.clip(depth_m[valid] / max_depth_m * 255.0, 0, 255).astype(np.uint8)
    return cv2.applyColorMap(disp, cv2.COLORMAP_JET)


def detect_visible_object(rgb_bgr, depth_m, hsv_low, hsv_high, min_area_px=500, max_depth_m=2.0):
    hsv = cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array(hsv_low, dtype=np.uint8), np.array(hsv_high, dtype=np.uint8))
    valid_depth = np.isfinite(depth_m) & (depth_m > 0.05) & (depth_m < float(max_depth_m))
    mask = mask & valid_depth.astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return False, {"area_px": 0}, rgb_bgr
    contour = max(contours, key=cv2.contourArea)
    area = float(cv2.contourArea(contour))
    overlay = rgb_bgr.copy()
    x, y, w, h = cv2.boundingRect(contour)
    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 255), 2)
    roi_depth = depth_m[y:y+h, x:x+w]
    valid = roi_depth[np.isfinite(roi_depth) & (roi_depth > 0)]
    median_depth = float(np.median(valid)) if valid.size else None
    return area >= float(min_area_px), {"area_px": area, "bbox": [x, y, w, h], "median_depth_m": median_depth}, overlay


Run the viewer. The default HSV range detects many red objects; tune the sliders for the object used in your exercise.


In [ ]:
h_low = widgets.IntSlider(value=0, min=0, max=179, description="H low")
h_high = widgets.IntSlider(value=12, min=0, max=179, description="H high")
s_low = widgets.IntSlider(value=80, min=0, max=255, description="S low")
s_high = widgets.IntSlider(value=255, min=0, max=255, description="S high")
v_low = widgets.IntSlider(value=50, min=0, max=255, description="V low")
v_high = widgets.IntSlider(value=255, min=0, max=255, description="V high")
min_area = widgets.IntSlider(value=500, min=50, max=10000, step=50, description="Area")
max_depth = widgets.FloatSlider(value=2.0, min=0.2, max=6.0, step=0.1, description="Depth m")
refresh = widgets.Button(description="Refresh Frame", button_style="success")
rgb_img = widgets.HTML(value="")
depth_img = widgets.HTML(value="")
status = widgets.HTML(value="")


def update(_=None):
    try:
        frame = robot.get_rgbd(timeout=2.0)
        rgb_bgr = frame["rgb_bgr"]
        depth_m = frame["depth_m"]
        visible, info, overlay = detect_visible_object(
            rgb_bgr,
            depth_m,
            (h_low.value, s_low.value, v_low.value),
            (h_high.value, s_high.value, v_high.value),
            min_area.value,
            max_depth.value,
        )
        depth_vis = colorize_depth(depth_m, max_depth.value)
        rgb_img.value = f'<img src="{jpeg_data_url(overlay)}" style="max-width:100%;"/>'
        depth_img.value = f'<img src="{jpeg_data_url(depth_vis)}" style="max-width:100%;"/>'
        status.value = f"visible={visible} info={info} source={frame['source']} center_depth={frame['center_depth_m']}"
    except Exception as exc:
        status.value = f"Frame update failed: {exc}"

refresh.on_click(update)
update()
display(widgets.VBox([
    widgets.HBox([h_low, h_high, s_low, s_high, v_low, v_high]),
    widgets.HBox([min_area, max_depth, refresh]),
    status,
    widgets.HBox([rgb_img, depth_img]),
]))


## Image segmentation and end-effector detection

This cell segments the RGBD frame by HSV and depth. Use the object controls for the table object and the end-effector controls for the robot hand or a colored marker on it.


In [ ]:
def segment_hsv_depth(rgb_bgr, depth_m, hsv_low, hsv_high, *, min_area_px=300, max_depth_m=2.5, roi=None, label="segment", color=(0, 255, 255)):
    hsv = cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array(hsv_low, dtype=np.uint8), np.array(hsv_high, dtype=np.uint8))
    valid_depth = np.isfinite(depth_m) & (depth_m > 0.05) & (depth_m < float(max_depth_m))
    mask = mask & valid_depth.astype(np.uint8) * 255
    if roi is not None:
        x0, y0, x1, y1 = roi
        keep = np.zeros_like(mask)
        keep[max(0, y0):min(mask.shape[0], y1), max(0, x0):min(mask.shape[1], x1)] = 255
        mask = mask & keep
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detections = []
    overlay = rgb_bgr.copy()
    for contour in sorted(contours, key=cv2.contourArea, reverse=True):
        area = float(cv2.contourArea(contour))
        if area < float(min_area_px):
            continue
        x, y, w, h = cv2.boundingRect(contour)
        roi_depth = depth_m[y:y + h, x:x + w]
        valid = roi_depth[np.isfinite(roi_depth) & (roi_depth > 0)]
        median_depth = float(np.median(valid)) if valid.size else None
        cx = int(x + w / 2)
        cy = int(y + h / 2)
        detections.append({
            "label": label,
            "bbox_xywh": [int(x), int(y), int(w), int(h)],
            "center_px": [cx, cy],
            "area_px": area,
            "median_depth_m": median_depth,
        })
        cv2.rectangle(overlay, (x, y), (x + w, y + h), color, 2)
        cv2.circle(overlay, (cx, cy), 5, color, -1)
        cv2.putText(overlay, label, (x, max(16, y - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return detections, mask, overlay


def mask_data_url(mask):
    return jpeg_data_url(cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR))

obj_h_low = widgets.IntSlider(value=0, min=0, max=179, description="Obj H low")
obj_h_high = widgets.IntSlider(value=12, min=0, max=179, description="Obj H high")
obj_s_low = widgets.IntSlider(value=80, min=0, max=255, description="Obj S low")
obj_v_low = widgets.IntSlider(value=50, min=0, max=255, description="Obj V low")
ee_h_low = widgets.IntSlider(value=0, min=0, max=179, description="EE H low")
ee_h_high = widgets.IntSlider(value=179, min=0, max=179, description="EE H high")
ee_s_low = widgets.IntSlider(value=0, min=0, max=255, description="EE S low")
ee_s_high = widgets.IntSlider(value=255, min=0, max=255, description="EE S high")
ee_v_low = widgets.IntSlider(value=0, min=0, max=255, description="EE V low")
ee_v_high = widgets.IntSlider(value=90, min=0, max=255, description="EE V high")
seg_min_area = widgets.IntSlider(value=300, min=50, max=8000, step=50, description="Area")
seg_depth = widgets.FloatSlider(value=2.5, min=0.2, max=6.0, step=0.1, description="Depth m")
seg_refresh = widgets.Button(description="Segment Frame", button_style="success")
seg_overlay_img = widgets.HTML(value="")
obj_mask_img = widgets.HTML(value="")
ee_mask_img = widgets.HTML(value="")
seg_status = widgets.Textarea(layout=widgets.Layout(width="100%", height="180px"), disabled=True)


def update_segmentation(_=None):
    try:
        frame = robot.get_rgbd(timeout=2.0)
        rgb_bgr = frame["rgb_bgr"]
        depth_m = frame["depth_m"]
        h, w = depth_m.shape[:2]
        obj_dets, obj_mask, obj_overlay = segment_hsv_depth(
            rgb_bgr,
            depth_m,
            (obj_h_low.value, obj_s_low.value, obj_v_low.value),
            (obj_h_high.value, 255, 255),
            min_area_px=seg_min_area.value,
            max_depth_m=seg_depth.value,
            label="object",
            color=(0, 255, 255),
        )
        ee_dets, ee_mask, _ = segment_hsv_depth(
            obj_overlay,
            depth_m,
            (ee_h_low.value, ee_s_low.value, ee_v_low.value),
            (ee_h_high.value, ee_s_high.value, ee_v_high.value),
            min_area_px=seg_min_area.value,
            max_depth_m=seg_depth.value,
            roi=(0, int(h * 0.35), w, h),
            label="end_effector",
            color=(255, 0, 255),
        )
        _, _, overlay = segment_hsv_depth(
            obj_overlay,
            depth_m,
            (ee_h_low.value, ee_s_low.value, ee_v_low.value),
            (ee_h_high.value, ee_s_high.value, ee_v_high.value),
            min_area_px=seg_min_area.value,
            max_depth_m=seg_depth.value,
            roi=(0, int(h * 0.35), w, h),
            label="end_effector",
            color=(255, 0, 255),
        )
        seg_overlay_img.value = f'<img src="{jpeg_data_url(overlay)}" style="max-width:100%;"/>'
        obj_mask_img.value = f'<img src="{mask_data_url(obj_mask)}" style="max-width:100%;"/>'
        ee_mask_img.value = f'<img src="{mask_data_url(ee_mask)}" style="max-width:100%;"/>'
        seg_status.value = json.dumps({"objects": obj_dets[:5], "end_effectors": ee_dets[:5], "source": frame["source"]}, indent=2)
    except Exception as exc:
        seg_status.value = f"Segmentation failed: {exc}"

seg_refresh.on_click(update_segmentation)
display(widgets.VBox([
    widgets.HBox([obj_h_low, obj_h_high, obj_s_low, obj_v_low]),
    widgets.HBox([ee_h_low, ee_h_high, ee_s_low, ee_s_high, ee_v_low, ee_v_high]),
    widgets.HBox([seg_min_area, seg_depth, seg_refresh]),
    seg_status,
    widgets.HBox([seg_overlay_img, obj_mask_img, ee_mask_img]),
]))
